In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from utils.bronze_table_airports import process_bronze_airports_table  # 或你定义的同等函数名
from utils.silver_table_airports import process_silver_airports_table

In [2]:
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()                            

spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/26 06:37:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Bronze Table

In [3]:
bronze_airport = "datamart/bronze/airport/"

if not os.path.exists(bronze_airport):
    os.makedirs(bronze_airport)

In [4]:
process_bronze_airports_table("data/airport-frequencies.csv", bronze_airport, spark)
process_bronze_airports_table("data/airports.csv", bronze_airport, spark)

[BRONZE] data/airport-frequencies.csv rows: 30165


saved to: datamart/bronze/airport/airport-frequencies
[BRONZE] data/airports.csv rows: 83748


saved to: datamart/bronze/airport/airports


DataFrame[id: int, ident: string, type: string, name: string, latitude_deg: double, longitude_deg: double, elevation_ft: int, continent: string, iso_country: string, iso_region: string, municipality: string, scheduled_service: string, icao_code: string, iata_code: string, gps_code: string, local_code: string, home_link: string, wikipedia_link: string, keywords: string, _ingest_ts: timestamp, _source_file: string, _ingest_date: date]

## Silver Table

In [5]:
silver_airport = "datamart/silver/airport/"

if not os.path.exists(silver_airport):
    os.makedirs(silver_airport)


In [14]:
process_silver_airports_table(
    bronze_airport_directory=bronze_airport,      # e.g. "datamart/bronze/airport"
    silver_airport_directory=silver_airport,      # e.g. "datamart/silver/airport"
    spark=spark
    # freq_type_whitelist=["TWR","APP","ATIS","GND"],   # 想要所有频率就传 None
    # scheduled_only=None                               # 只要有定期航班机场就 True；不过滤就 None
)


[SILVER] US airports wide -> datamart/silver/airport/US_airports


{'wide_path': 'datamart/silver/airport/US_airports', 'subset_path': None}

In [19]:
core_path = process_silver_airports_table(
    bronze_airport_directory=bronze_airport,      # e.g. "datamart/bronze/airport"
    silver_airport_directory=silver_airport,      # e.g. "datamart/silver/airport"
    spark=spark,
    freq_type_whitelist=None,             # e.g. ["TWR","APP","A/D","ATIS","AWOS","GND"]; None = no filter
    scheduled_only=None,                  # True/False to filter scheduled_service; None = no filter
    iata_targets=["JFK","LGA","EWR"],                    # e.g. ["JFK","LGA","EWR"]; None = don't make subset
    wide_subdir="US_airports",            # 全美 US 宽表子目录
    subset_subdir="new_york"              # 纽约三场子表子目录
)

df_us  = spark.read.parquet("datamart/silver/airport/US_airports")
print("rows:", df_us.count())
df_us.printSchema()

[SILVER] US airports wide -> datamart/silver/airport/US_airports
[SILVER] subset (EWR,JFK,LGA) -> datamart/silver/airport/new_york
+----------+-----+---------+------------------------------------+----------+
|airport_id|ident|iata_code|name                                |iso_region|
+----------+-----+---------+------------------------------------+----------+
|3622      |KJFK |JFK      |John F Kennedy International Airport|US-NY     |
|3521      |KEWR |EWR      |Newark Liberty International Airport|US-NJ     |
|3643      |KLGA |LGA      |LaGuardia Airport                   |US-NY     |
+----------+-----+---------+------------------------------------+----------+

rows: 4005
root
 |-- airport_id: integer (nullable = true)
 |-- ident: string (nullable = true)
 |-- name: string (nullable = true)
 |-- airport_type: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- continent: string (

In [21]:
df_newyork = spark.read.parquet("datamart/silver/airport/new_york")
print("rows:", df_newyork.count())
df_newyork.printSchema()

rows: 3
root
 |-- airport_id: integer (nullable = true)
 |-- ident: string (nullable = true)
 |-- name: string (nullable = true)
 |-- airport_type: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- scheduled_service_bool: boolean (nullable = true)
 |-- _ident_match: integer (nullable = true)
 |-- has_geo: boolean (nullable = true)
 |-- freq_tower: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- freq_ground: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- freq_approach: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- freq_departure: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- freq_approach_

In [23]:
df_us.toPandas()

,airport_id,ident,name,airport_type,municipality,iso_country,iso_region,continent,latitude_deg,longitude_deg,...,has_approach,has_departure,has_approach_departure,has_atis,has_awos,has_ctaf_unicom,has_clearance_delivery,has_fss,has_center,has_other
0,3356,KABE,Lehigh Valley International Airport,medium_airport,Allentown,US,US-PA,NA,40.651773,-75.442797,...,False,True,False,False,False,True,True,False,False,True
1,3357,KABI,Abilene Regional Airport,medium_airport,Abilene,US,US-TX,NA,32.411301,-99.681900,...,False,True,False,False,False,True,False,False,False,True
2,3358,KABR,Aberdeen Regional Airport,medium_airport,Aberdeen,US,US-SD,NA,45.449100,-98.421799,...,False,False,False,False,False,True,False,False,False,True
3,3359,KABY,Southwest Georgia Regional Airport,medium_airport,Albany,US,US-GA,NA,31.532946,-84.196215,...,False,False,False,False,False,True,False,False,False,True
4,3360,KACK,Nantucket Memorial Airport,medium_airport,Nantucket,US,US-MA,NA,41.253101,-70.060204,...,False,True,False,False,False,True,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4000,329459,US-0986,Dead Cow Lakebed Airstrip,small_airport,"Herlong, CA",US,US-NV,NA,40.147755,-119.907335,...,False,False,False,False,False,True,False,False,False,False
4001,336651,US-1858,Ibex / Tule Valley Hardpan Airstrip,small_airport,Garrison,US,US-UT,NA,38.949020,-113.377060,...,False,False,False,False,False,True,False,False,False,False
4002,349666,US-5712,Kenyon Airstrip,small_airport,Burley,US,US-ID,NA,42.436306,-113.856384,...,False,False,False,False,False,True,False,False,False,False
4003,354717,US-6836,Greenleaf Air Ranch Airport,small_airport,Greenleaf,US,US-ID,NA,43.681793,-116.826618,...,False,False,False,False,False,True,False,False,False,False


In [22]:
df_newyork.toPandas()

,airport_id,ident,name,airport_type,municipality,iso_country,iso_region,continent,latitude_deg,longitude_deg,...,has_atis,has_awos,has_ctaf_unicom,has_clearance_delivery,has_fss,has_center,has_other,iata_code,ident_bz,ident_up
0,3521,KEWR,Newark Liberty International Airport,large_airport,Newark,US,US-NJ,NA,40.692501,-74.168701,...,False,False,True,True,False,False,True,EWR,KEWR,KEWR
1,3622,KJFK,John F Kennedy International Airport,large_airport,New York,US,US-NY,NA,40.639447,-73.779317,...,False,False,True,True,False,False,True,JFK,KJFK,KJFK
2,3643,KLGA,LaGuardia Airport,large_airport,New York,US,US-NY,NA,40.777199,-73.872597,...,False,False,True,True,False,False,True,LGA,KLGA,KLGA


In [25]:
# 只保留我们关心的列
has_cols = [c for c in df.columns if c.startswith("has_")]
base_cols = ["airport_id","ident","name","iata_code","iso_region"]
eda = df_newyork.select(*base_cols, *has_cols)

print("Airports:", eda.select("ident","iata_code","name").toPandas())
print("Facilities columns:", has_cols)
eda.show(truncate=False)


Airports:   ident iata_code                                  name
0  KEWR       EWR  Newark Liberty International Airport
1  KJFK       JFK  John F Kennedy International Airport
2  KLGA       LGA                     LaGuardia Airport
Facilities columns: ['has_geo', 'has_tower', 'has_ground', 'has_approach', 'has_departure', 'has_approach_departure', 'has_atis', 'has_awos', 'has_ctaf_unicom', 'has_clearance_delivery', 'has_fss', 'has_center', 'has_other']
+----------+-----+------------------------------------+---------+----------+-------+---------+----------+------------+-------------+----------------------+--------+--------+---------------+----------------------+-------+----------+---------+
|airport_id|ident|name                                |iata_code|iso_region|has_geo|has_tower|has_ground|has_approach|has_departure|has_approach_departure|has_atis|has_awos|has_ctaf_unicom|has_clearance_delivery|has_fss|has_center|has_other|
+----------+-----+------------------------------------+--

In [27]:
# 想展示的设施（顺序可改）
core_has = [
    "has_tower","has_ground","has_approach","has_departure","has_approach_departure",
    "has_atis","has_awos","has_ctaf_unicom","has_clearance_delivery","has_fss","has_center","has_other"
]

# 1) 取需要的列
base_cols = [c for c in ["ident","iata_code","name"] if c in df_newyork.columns]
df_view = df_newyork.select(*base_cols, *core_has)

# 2) 到 pandas 并把布尔转成 ✓ / 空白
pdf = df_view.toPandas().copy()

col_name_map = {
    "has_tower": "Tower",
    "has_ground": "Ground",
    "has_approach": "Approach",
    "has_departure": "Departure",
    "has_approach_departure": "Approach/Departure",
    "has_atis": "ATIS",
    "has_awos": "AWOS/ASOS",
    "has_ctaf_unicom": "CTAF/UNICOM",
    "has_clearance_delivery": "Clearance Delivery",
    "has_fss": "FSS/RCO",
    "has_center": "Center",
    "has_other": "Other"
}

for c in core_has:
    nice = col_name_map[c]
    pdf[nice] = pdf[c].map(lambda x: "✓" if bool(x) else "X")

# 3) 清理/重命名基础列 & 排列顺序
pdf = (pdf
       .drop(columns=core_has)
       .rename(columns={"ident":"IDENT", "iata_code":"IATA", "name":"Airport"})
       .set_index("IATA")
       .sort_index()
      )

nice_order = ["Airport","IDENT"] + list(col_name_map.values())
pdf = pdf[[c for c in nice_order if c in pdf.columns]]

# 4) 展示（Jupyter 会渲染成漂亮表格）
pdf

,Airport,IDENT,Tower,Ground,Approach,Departure,Approach/Departure,ATIS,AWOS/ASOS,CTAF/UNICOM,Clearance Delivery,FSS/RCO,Center,Other
IATA,,,,,,,,,,,,,,
EWR,Newark Liberty International Airport,KEWR,X,X,X,X,X,X,X,✓,✓,X,X,✓
JFK,John F Kennedy International Airport,KJFK,X,X,X,X,X,X,X,✓,✓,X,X,✓
LGA,LaGuardia Airport,KLGA,X,X,X,X,X,X,X,✓,✓,X,X,✓
